# 🚀 BATCH FACE SWAP STUDIO - HOÁN ĐỔI MẶT HÀNG LOẠT QUA GOOGLE DRIVE
> **Quy trình 2 bước cực kỳ dễ dàng:**
> 1. **Bấm chạy Ô 1**: Để kết nối Google Drive, cài đặt tăng tốc CUDA GPU và tự động tạo sẵn các thư mục.
> 2. **Vào Google Drive thả ảnh & video vào thư mục**, sau đó quay lại **bấm chạy Ô 2**!

In [ ]:
#@title 📦 Ô 1: CÀI ĐẶT MÔI TRƯỜNG & TẠO THƯ MỤC GOOGLE DRIVE (Chạy 1 lần đầu)
#@markdown > Bấm nút ▶️ ở ô này để kết nối Google Drive, cấu hình GPU Tesla T4 và tạo sẵn thư mục.

import os
import sys
import subprocess

print("=" * 65)
print("📦 BẮT ĐẦU CÀI ĐẶT MÔI TRƯỜNG BATCH FACE SWAP CHO GOOGLE COLAB")
print("=" * 65)

# 1. Kết nối Google Drive
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    print("🔗 Đang kết nối Google Drive...")
    drive.mount('/content/drive')
    print("✓ Đã gắn kết Google Drive thành công!")
else:
    print("✓ Google Drive đã sẵn sàng!")

base_dir = "/content/drive/MyDrive/AI_Colab_Cache/BatchFaceSwap"
mat_moi_dir = os.path.join(base_dir, "mat_moi")
nguoi_trong_clip_dir = os.path.join(base_dir, "nguoi_trong_clip")
target_dir = os.path.join(base_dir, "target_videos")
output_dir = os.path.join(base_dir, "output_videos")
models_dir = os.path.join(base_dir, "models")

for d in [mat_moi_dir, nguoi_trong_clip_dir, target_dir, output_dir, models_dir]:
    os.makedirs(d, exist_ok=True)

print("\n📁 ĐÃ TỰ ĐỘNG TẠO SẴN CÁC THƯ MỤC TRÊN GOOGLE DRIVE CỦA BẠN:")
print(f"  1. 📁 {mat_moi_dir}")
print("     👉 Thả ảnh chân dung MẶT MỚI muốn đắp vào đây")
print(f"  2. 📁 {nguoi_trong_clip_dir}")
print("     👉 (Tùy chọn) Chụp màn hình mặt người trong clip cần đổi nếu clip có 2-3 người")
print(f"  3. 📁 {target_dir}")
print("     👉 Thả các file VIDEO cần đổi mặt vào đây")
print(f"  4. 📁 {output_dir}")
print("     👉 Nơi lưu toàn bộ video thành phẩm sau khi swap xong")

# 2. Cài đặt thư viện & CUDA GPU
print("\n⏳ Đang cài đặt thư viện và kích hoạt tăng tốc GPU Tesla T4...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "opencv-python-headless", "tqdm", "insightface"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu", "--extra-index-url", "https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], check=False)

# 3. Tải trước model Inswapper 128 vào Google Drive Cache
swapper_path = os.path.join(models_dir, "inswapper_128.onnx")
if not os.path.exists(swapper_path) or os.path.getsize(swapper_path) < 100000000:
    print("⏳ Đang tải mô hình Inswapper 128 ONNX (~529MB) về Google Drive (chỉ tải 1 lần duy nhất)...")
    download_url = "https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx"
    subprocess.run(['curl', '-L', download_url, '-o', swapper_path], check=True)
    print("✓ Đã lưu Inswapper vào Google Drive!")
else:
    print("✓ Mô hình Inswapper 128 đã có sẵn trong Google Drive!")

print("\n" + "=" * 65)
print("🎉 CÀI ĐẶT HOÀN TẤT 100%!")
print("👉 BÂY GIỜ BẠN HÃY VÀO GOOGLE DRIVE THẢ ẢNH VÀ VIDEO VÀO CÁC THƯ MỤC:")
print(f"   • Thả ảnh mặt mới vào thư mục : mat_moi")
print(f"   • Thả video cần đổi vào       : target_videos")
print(f"   • (Nếu clip nhiều người) chụp ảnh mặt trong clip thả vào : nguoi_trong_clip")
print("👉 SAU ĐÓ XUỐNG Ô 2 BÊN DƯỚI BẤM NÚT ▶️ ĐỂ BẮT ĐẦU CHẠY!")
print("=" * 65)


In [ ]:
#@title 🚀 Ô 2: BẢNG TÙY CHỌN & BẤM CHẠY ĐỔI MẶT HÀNG LOẠT
#@markdown ### ⚡ 1. Chế độ tốc độ GPU Tesla T4:
TOC_DO_GPU = "🔥 Turbo Kịch Khung (~35-40 FPS)" #@param ["🔥 Turbo Kịch Khung (~35-40 FPS)", "Tiêu chuẩn (~20 FPS)"]

#@markdown ### 🎛️ 2. Làm nét mặt (Tắt: Siêu Tốc ~35-40 FPS chỉ ~50s/clip | Bật: ~5 FPS nét sâu):
BAT_LAM_NET_MAT = False #@param {type:"boolean"}

#@markdown ### 👥 3. Chế độ chọn khuôn mặt (khi không dùng ảnh mẫu trong nguoi_trong_clip):
CHE_DO_CHON_MAT = "Mặt nhân vật chính (Lớn nhất)" #@param ["Mặt nhân vật chính (Lớn nhất)", "Đổi tất cả các mặt trong video"]

# Chuyển đổi tùy chọn sang tham số lệnh
speed_flag = "turbo" if "turbo" in TOC_DO_GPU.lower() else "standard"
enhance_flag = "true" if BAT_LAM_NET_MAT else "false"
selector_flag = "all" if "tất cả" in CHE_DO_CHON_MAT.lower() else "largest"

!curl -s -L "https://raw.githubusercontent.com/nviethiep55-glitch/my-ai-studio-colab/main/batch_face_swap.py?v=$(date +%s)" -o /content/batch_face_swap.py
!python -u /content/batch_face_swap.py --speed $speed_flag --enhance $enhance_flag --selector $selector_flag
